In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [2]:
from setproctitle import setproctitle
setproctitle("dqn_vs_alphazero")

In [3]:
import sys
sys.path.append('..')

In [4]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

2025-01-26 11:16:48.566241: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-26 11:16:48.566273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-26 11:16:48.567223: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-26 11:16:48.572153: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-26 11:16:49.383607: W tensorflow/compiler/tf2

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import numpy as np
from tqdm import trange
from UltimateTicTacToeEnvSelfPlay import UltimateTicTacToeEnvSelfPlay
from DQNAgent import DQNAgent
from AlphaZeroImproved import AlphaZero

In [7]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
dqn_agent = DQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DQN")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    dqn_states = np.array([env.to_state()[0] for env in envs])
    dqn_available_actions = np.array([env.to_state()[1] for env in envs])
    dqn_actions = dqn_agent.choose_action(dqn_states, dqn_available_actions, True)
    dqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, dqn_reward[i], game_finished[i], _  = envs[i].step(dqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = dqn_reward[i]
    
    alphazero_agent = AlphaZero(envs, input_dim=state_space_shape, action_dim=action_space_size, simulations=200, actor_path="../models/Thesis_PPO/actor", critic_path="../models/Thesis_PPO/critic")
    alphazero_states = np.array([env.to_state()[0] for env in envs])
    alphazero_available_actions = np.array([env.to_state()[1] for env in envs])
    alphazero_actions = alphazero_agent.play()
    alphazero_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, alphazero_reward[i], game_finished[i], _  = envs[i].step(alphazero_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -alphazero_reward[i]

    alphazero_agent.update_tree_with_move(alphazero_actions)
    print("Both players have done a move.")

win_as_first_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_first_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as first player: ", win_as_first_player)
print("draw rate as first player: ", draw_as_first_player)

2025-01-26 11:16:51.359490: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-01-26 11:16:51.359787: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-01-26 11:16:51.360013: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Models loaded from memory
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
5  out of  500
Both players have done a move.
8  out of  500
Both players have done a move.
13  out of  500
Both players have done a move.
38  out of  500
Both players have done a move.
60  out of  500
Both players have done a move.
93  out of  500
Both players have done a move.
130  out of  500
Both players have done a move.
18

In [8]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
dqn_agent = DQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DQN")
alphazero_agent = AlphaZero(envs, input_dim=state_space_shape, action_dim=action_space_size, simulations=200, actor_path="../models/Thesis_PPO/actor", critic_path="../models/Thesis_PPO/critic")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    alphazero_states = np.array([env.to_state()[0] for env in envs])
    alphazero_available_actions = np.array([env.to_state()[1] for env in envs])
    alphazero_actions = alphazero_agent.play()
    alphazero_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, alphazero_reward[i], game_finished[i], _  = envs[i].step(alphazero_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -alphazero_reward[i]
                
    dqn_states = np.array([env.to_state()[0] for env in envs])
    dqn_available_actions = np.array([env.to_state()[1] for env in envs])
    dqn_actions = dqn_agent.choose_action(dqn_states, dqn_available_actions, True)
    dqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, dqn_reward[i], game_finished[i], _  = envs[i].step(dqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = dqn_reward[i]
    alphazero_agent.update_tree_with_move(alphazero_actions)
    alphazero_agent.update_tree_with_move(dqn_actions)
    print("Both players have done a move.")

win_as_second_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_second_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as second player: ", win_as_second_player)
print("draw rate as second player: ", draw_as_second_player)

Models loaded from memory
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
5  out of  500
Both players have done a move.
10  out of  500
Both players have done a move.
17  out of  500
Both players have done a move.
26  out of  500
Both players have done a move.
44  out of  500
Both players have done a move.
71  out of  500
Both players have done a move.
107

In [9]:
total_win = (win_as_first_player+win_as_second_player)/2
total_draw = (draw_as_first_player+draw_as_second_player)/2
print("Total win rate: ", total_win)
print("Total draw rate: ", total_draw)
print("Total loss: ", 1-total_draw-total_win)

Total win rate:  0.10400000000000001
Total draw rate:  0.053000000000000005
Total loss:  0.843
